In [2]:
from rdkit import Chem

from mordred import Calculator
from mordred import descriptors

from mendeleev import element

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

from xgboost import XGBRegressor

In [3]:
# =========================================================
# LOAD DATA
# =========================================================

df = pd.read_csv(
    "Reduced_Metallophilic_Dataset.csv"
)

# =========================================================
# HIDDEN TEST SYSTEMS
# =========================================================
df["HX_nearest"] = (
    df["HX_nearest"]
    .fillna(0)
)
hammett_map = {

    "NH2": -0.66,
    "OH" : -0.37,
    "CH3": -0.17,

    "H"  : 0.00,

    "F"  : 0.06,

    "CF3": 0.54,
    "NO2": 0.78
}

df["Hammett_sigma"] = (
    df["R"]
    .map(hammett_map)
)
hidden_cu = [

]

train_df = df[
    ~df["dimer"].isin(
        hidden_cu
    )
].copy()

# =========================================================
# ML DESCRIPTORS
# =========================================================

ml_columns = [

    "Metal_atomic_number",
    "Metal_atomic_radius",
    "Metal_electronegativity",
    "Metal_polarizability",
    "Metal_ionization_energy",

    "X_atomic_number",
    "X_atomic_radius",
    "X_electronegativity",
    "X_polarizability",
    "X_ionization_energy",

    "nAromAtom",
    "nN",
    "nO",
    "nF",
    "nAtom",

    "Hammett_sigma"
]

# =========================================================
# TRAIN MODEL
# =========================================================

X_train = train_df[
    ml_columns
]

y_train = train_df[
    "E"
]

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(

    scaler.fit_transform(
        X_train
    ),

    columns=ml_columns

)

final_model = XGBRegressor(

    n_estimators=200,

    learning_rate=0.05,

    max_depth=4,

    random_state=42

)

final_model.fit(

    X_train_scaled,

    y_train

)

print(
    "Model trained."
)

Model trained.


In [4]:
Y_smiles = {

    # -----------------------------------------------------
    # PYRIDINE FAMILY
    # -----------------------------------------------------

    "4-H-Py":
    "n1ccccc1",

    "4-F-Py":
    "n1ccc(F)cc1",

    "4-CH3-Py":
    "n1ccc(C)cc1",

    "4-OH-Py":
    "n1ccc(O)cc1",

    "4-NH2-Py":
    "n1ccc(N)cc1",

    "4-NO2-Py":
    "n1ccc([N+](=O)[O-])cc1",

    "4-CF3-Py":
    "n1ccc(C(F)(F)F)cc1",

    # -----------------------------------------------------
    # CN FAMILY
    # -----------------------------------------------------

    "H-CN":
    "C#N",

    "F-CN":
    "FC#N",

    "CH3-CN":
    "CC#N",

    "OH-CN":
    "OC#N",

    "NH2-CN":
    "NC#N",

    "NO2-CN":
    "O=[N+]([O-])C#N",

    "CF3-CN":
    "FC(F)(F)C#N"
}
hammett_map = {

    "NH2": -0.66,
    "OH" : -0.37,
    "CH3": -0.17,

    "H"  : 0.00,

    "F"  : 0.06,

    "CF3": 0.54,
    "NO2": 0.78
}

In [5]:
def extract_periodic_descriptors(symbol):

    el = element(symbol)

    descriptors = {

        "atomic_number": el.atomic_number,

        "atomic_radius": el.atomic_radius,

        "electronegativity": el.en_pauling,

        "polarizability": el.dipole_polarizability,

        "ionization_energy": el.ionenergies[1]

    }

    return descriptors

In [6]:
# =========================================================
# METALLOPHILIC ENERGY PREDICTOR
# =========================================================

def predict_energy(
    Metal,
    Ligand,
    X,
    R
):

    # =====================================================
    # INPUT VALIDATION
    # =====================================================

    allowed_metals = [
        "Cu",
        "Ag",
        "Au"
    ]

    allowed_ligands = [
        "Py",
        "CN"
    ]

    allowed_X = [
        "F",
        "Cl",
        "Br",
        "I",
        "CN"
    ]

    allowed_R = [
        "NH2",
        "OH",
        "CH3",
        "H",
        "F",
        "CF3",
        "NO2"
    ]

    if Metal not in allowed_metals:
        raise ValueError(f"Invalid Metal: {Metal}")

    if Ligand not in allowed_ligands:
        raise ValueError(f"Invalid Ligand: {Ligand}")

    if X not in allowed_X:
        raise ValueError(f"Invalid X: {X}")

    if R not in allowed_R:
        raise ValueError(f"Invalid R: {R}")

    # =====================================================
    # GET SMILES
    # =====================================================

    if Ligand == "Py":

        ligand_key = f"4-{R}-Py"

    else:

        ligand_key = f"{R}-CN"

    smiles = Y_smiles[ligand_key]

    # =====================================================
    # GENERATE MORDRED DESCRIPTORS
    # =====================================================

    mol = Chem.MolFromSmiles(smiles)

    calc = Calculator(
        descriptors,
        ignore_3D=True
    )

    mordred_df = calc.pandas([mol])

    # =====================================================
    # LIGAND DESCRIPTORS
    # =====================================================

    ligand_features = {

        "nAromAtom":
            float(mordred_df["nAromAtom"].iloc[0]),

        "nN":
            float(mordred_df["nN"].iloc[0]),

        "nO":
            float(mordred_df["nO"].iloc[0]),

        "nF":
            float(mordred_df["nF"].iloc[0]),

        "nAtom":
            float(mordred_df["nAtom"].iloc[0])
    }

    # =====================================================
    # METAL DESCRIPTORS
    # =====================================================

    metal_desc = extract_periodic_descriptors(
        Metal
    )

    metal_features = {

        "Metal_atomic_number":
            metal_desc["atomic_number"],

        "Metal_atomic_radius":
            metal_desc["atomic_radius"],

        "Metal_electronegativity":
            metal_desc["electronegativity"],

        "Metal_polarizability":
            metal_desc["polarizability"],

        "Metal_ionization_energy":
            metal_desc["ionization_energy"]
    }

    # =====================================================
    # X DESCRIPTORS
    # =====================================================

    x_symbol = (
        "C"
        if X == "CN"
        else X
    )

    x_desc = extract_periodic_descriptors(
        x_symbol
    )

    x_features = {

        "X_atomic_number":
            x_desc["atomic_number"],

        "X_atomic_radius":
            x_desc["atomic_radius"],

        "X_electronegativity":
            x_desc["electronegativity"],

        "X_polarizability":
            x_desc["polarizability"],

        "X_ionization_energy":
            x_desc["ionization_energy"]
    }

    # =====================================================
    # HAMMETT DESCRIPTOR
    # =====================================================

    hammett_sigma = hammett_map[R]

    # =====================================================
    # FEATURE VECTOR
    # =====================================================

    feature_dict = {

        **metal_features,

        **x_features,

        **ligand_features,

        "Hammett_sigma":
            hammett_sigma
    }

    feature_df = pd.DataFrame(
        [feature_dict]
    )

    feature_df = feature_df[
        ml_columns
    ]

    # =====================================================
    # SCALE FEATURES
    # =====================================================

    feature_scaled = scaler.transform(
        feature_df
    )

    # =====================================================
    # PREDICT ENERGY
    # =====================================================

    predicted_energy = float(

        final_model.predict(
            feature_scaled
        )[0]

    )

    # =====================================================
    # OUTPUT
    # =====================================================

    print("\n")
    print("=" * 60)
    print("METALLOPHILIC INTERACTION ENERGY PREDICTION")
    print("=" * 60)

    print(f"Metal    : {Metal}")
    print(f"Ligand   : {Ligand}")
    print(f"X        : {X}")
    print(f"R Group  : {R}")

    print("\n")

    print(
        f"Predicted Energy = "
        f"{predicted_energy:.3f} kcal/mol"
    )

    print("=" * 60)

    return predicted_energy

In [7]:
# =========================================================
# USER INPUT
# =========================================================

Metal = input(
    "Enter Metal (Cu/Ag/Au): "
).strip()

Ligand = input(
    "Enter Ligand (Py/CN): "
).strip()

X = input(
    "Enter X (F/Cl/Br/I/CN): "
).strip()

R = input(
    "Enter R Group (NH2/OH/CH3/H/F/CF3/NO2): "
).strip()

# =========================================================
# PREDICT ENERGY
# =========================================================

predict_energy(

    Metal=Metal,

    Ligand=Ligand,

    X=X,

    R=R

)

Enter Metal (Cu/Ag/Au): Cu
Enter Ligand (Py/CN): CN
Enter X (F/Cl/Br/I/CN): F
Enter R Group (NH2/OH/CH3/H/F/CF3/NO2): OH


100%|██████████| 1/1 [00:00<00:00,  1.01it/s]




METALLOPHILIC INTERACTION ENERGY PREDICTION
Metal    : Cu
Ligand   : CN
X        : F
R Group  : OH


Predicted Energy = -6.210 kcal/mol


-6.209934234619141

'Cu_53'